# 02. Dataset, Sampler, DataLoader, and Collate

This notebook explains the last part of the data pipeline:

```text
H5adSentenceDataset.__getitem__
  -> retrieve target perturbed counts
  -> map to a matched control cell
  -> return one cell row plus covariates

CellSetBatchSampler
  -> group perturb cells by (pert, cell_type, batch)
  -> pad each group to use_cell_set
  -> yield cells in blocks of use_cell_set

collate_fn
  -> stack rows
  -> normalize counts
  -> reshape to [outer_batch, use_cell_set, genes]
```

The key thing to verify is that each cell-set contains cells from one experimental condition.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start=None):
    cur = Path(start or Path.cwd()).resolve()
    for p in [cur, *cur.parents]:
        if (p / "src" / "data").exists() and (p / "configs").exists():
            return p
    raise RuntimeError("Could not find PerturbDiff repo root")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PERTURB_ROOT = PROJECT_ROOT / "data" / "replogle" / "perturb_data"
RUN_DIR = PROJECT_ROOT / "runs" / "data_pipeline_dataloader"
WANDB_DIR = PROJECT_ROOT / "wandb"
assert PERTURB_ROOT.exists(), "Run example/replogle/00_download_data.ipynb first."

In [ ]:
import numpy as np
import torch
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf

from src.common.utils import setup_loggings
from src.apps.sampling.sampling_setup import build_sampling_datamodule

overrides = [
    "path=trixie_path",
    f"path.tmp_dir={PERTURB_ROOT}",
    f"path.diffusion.save_dir={RUN_DIR}",
    f"path.wandb.logging_dir={WANDB_DIR}",
    "data=replogle_finetune",
    "data.normalize_counts=10",
    "data.pad_length=2000",
    "data.embed_key=X_hvg",
    "model.hidden_num=[2000,512]",
    "model.input_dim=2000",
    "optimization.micro_batch_size=16",
    "data.use_cell_set=8",
]

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "configs"), version_base=None):
    cfg = compose(config_name="rawdata_diffusion_sampling", overrides=overrides)
OmegaConf.resolve(cfg)
logger = setup_loggings(cfg)

dm = build_sampling_datamodule(cfg, logger)
dm.setup_dataset()

print("use_cell_set:", cfg.data.use_cell_set)
print("micro_batch_size:", dm.micro_batch_size)
print("outer cell-set batch after collate:", dm.micro_batch_size // cfg.data.use_cell_set)
print("splits:", dm.all_split_names)

## Inspect a Dataset Item

`H5adSentenceDataset.__getitem__` returns one cell. The dataloader later groups many such rows into a cell-set.

In [ ]:
dataset = dm.test_dataset
print("dataset class:", type(dataset).__name__)
print("dataset stage:", dataset.stage)
print("len(dataset):", len(dataset))
print("dataset_path_map:", dataset.dataset_path_map)
print("selected genes length:", len(dataset.selected_genes))

item = dataset[0]
names = ["counts", "mapped_counts", "gene_vars", "pert_var", "mapped_pert_var", "cell_type_var", "batch_var", "is_padded", "ds_name"]
for name, value in zip(names, item):
    if torch.is_tensor(value):
        print(f"{name:>18}: tensor shape={tuple(value.shape)} dtype={value.dtype}")
    elif isinstance(value, (list, tuple, np.ndarray)):
        print(f"{name:>18}: {type(value).__name__} len={len(value)} sample={list(value)[:5]}")
    else:
        print(f"{name:>18}: {value}")

## Decode One Item's Covariates

`pert_var`, `cell_type_var`, and `batch_var` are global dictionary ids. The metadata cache lets us map them back to labels.

In [ ]:
inv_pert = {v: k for k, v in dm.pert_dict.items()}
inv_cell = {v: k for k, v in dm.cell_type_dict.items()}
inv_batch = {v: k for k, v in dm.batch_dict.items()}

_, _, _, pert_var, mapped_pert_var, cell_type_var, batch_var, is_padded, ds_name = item
print("target pert:", inv_pert[int(pert_var)])
print("mapped/control pert:", inv_pert[int(mapped_pert_var)])
print("cell type:", inv_cell[int(cell_type_var)])
print("batch:", inv_batch[int(batch_var)])
print("is_padded:", is_padded)
print("ds_name:", ds_name)

## Inspect the Cell-Set Sampler

For perturb-seq, groups are keyed by `(pert_code, cell_type_code, batch_code)`. The sampler pads every group to a multiple of `use_cell_set`, then yields cells in blocks of that size.

In [ ]:
from src.data.sampler import CellSetBatchSampler

sampler = CellSetBatchSampler(dataset, shuffle=False, seed=dm.seed)
print("sampler length after padding:", len(sampler))
print("total padded cell slots:", sampler.total_samples)
print("dsname_ref:", sampler.dsname_ref)
print("all_block_indices shape:", sampler._all_block_indices.shape)
print("all_block_ignored shape:", sampler._all_block_ignored.shape)

nonempty_groups = []
for ds_name, groups in dataset.grouped_pert_data_indices.items():
    for key, val in groups.items():
        if len(val) > 0:
            nonempty_groups.append((ds_name, key, len(val)))
print("nonempty groups:", len(nonempty_groups))
print("first 10 groups:", nonempty_groups[:10])

## Pull Raw Sampler Blocks

Each block of 8 yielded triplets should come from one `(pert, cell_type, batch)` group. Padding slots are marked by `is_padded=1`.

In [ ]:
it = iter(sampler)
block = [next(it) for _ in range(cfg.data.use_cell_set)]
print("first sampler block:")
for triplet in block:
    print(" ", triplet)

decoded = []
for ds_name, local_idx, is_padded in block:
    cache = dataset.meta_cache._cache[dataset.dataset_path_map[ds_name]]
    pert = cache.pert_categories[cache.pert_codes[local_idx]]
    cell = cache.cell_type_categories[cache.cell_type_codes[local_idx]]
    batch = cache.batch_categories[cache.batch_codes[local_idx]]
    decoded.append((pert, cell, batch, is_padded))
print("\ndecoded block:")
for row in decoded:
    print(" ", row)
print("unique target conditions:", set((p, c, b) for p, c, b, _ in decoded))

## DataLoader and Collate Shapes

The dataloader uses `batch_size=micro_batch_size` on individual cells. Then `collate_fn` reshapes them into `micro_batch_size // use_cell_set` cell-sets.

In [ ]:
loaders = dm.val_dataloader()
validation_loader, test_loader = loaders
batch = next(iter(test_loader))

def describe(x):
    if torch.is_tensor(x):
        return f"Tensor shape={tuple(x.shape)} dtype={x.dtype}"
    if isinstance(x, np.ndarray):
        return f"ndarray shape={x.shape} dtype={x.dtype}"
    if isinstance(x, list):
        return f"list len={len(x)} first_type={type(x[0]).__name__ if x else 'empty'}"
    return f"{type(x).__name__}: {str(x)[:80]}"

for key in sorted(batch.keys()):
    print(f"{key:>18}: {describe(batch[key])}")

## Verify Covariate Consistency Inside Each Cell-Set

For Replogle, each non-padding cell in a set should share target perturbation, cell type, and batch. This is the invariant created by `group_by_three_keys(...)` plus `CellSetBatchSampler`.

In [ ]:
cov_pert = batch["cov_pert"]
cov_cell = batch["cov_celltype"]
cov_batch = batch["cov_batch"]
pad = batch["is_padded_list"].bool()

for set_idx in range(cov_pert.shape[0]):
    valid = ~pad[set_idx]
    print("\ncell-set", set_idx)
    print("  valid cells:", int(valid.sum()), "of", valid.numel())
    print("  cov_pert ids:", cov_pert[set_idx].tolist())
    print("  cov_celltype ids:", cov_cell[set_idx].tolist())
    print("  cov_batch ids:", cov_batch[set_idx].tolist())
    print("  unique valid pert ids:", torch.unique(cov_pert[set_idx][valid]).tolist())
    print("  unique valid cell ids:", torch.unique(cov_cell[set_idx][valid]).tolist())
    print("  unique valid batch ids:", torch.unique(cov_batch[set_idx][valid]).tolist())

## Target vs Control Expressions

`pert_emb` is the target perturbed cell expression. `cont_emb` is the mapped control expression, sampled from control cells with the same cell type and batch when possible.

In [ ]:
pert_emb = batch["pert_emb"]
cont_emb = batch["cont_emb"]
print("pert_emb shape:", tuple(pert_emb.shape))
print("cont_emb shape:", tuple(cont_emb.shape))
print("pert_emb stats mean/std/min/max:", float(pert_emb.mean()), float(pert_emb.std()), float(pert_emb.min()), float(pert_emb.max()))
print("cont_emb stats mean/std/min/max:", float(cont_emb.mean()), float(cont_emb.std()), float(cont_emb.min()), float(cont_emb.max()))
print("mean absolute target-control difference:", float((pert_emb - cont_emb).abs().mean()))

## Pipeline Summary

The Replogle dataloader yields batches with this structure:

```text
micro_batch_size = 16 individual cells before collate
use_cell_set = 8

after collate:
  pert_emb     [2, 8, 2000]
  cont_emb     [2, 8, 2000]
  cov_pert     [2, 8]
  cov_celltype [2, 8]
  cov_batch    [2, 8]
```

The inner `8` dimension is a cell-set drawn from one `(pert, cell_type, batch)` group. Padding slots are tracked by `is_padded_list` and removed during sampling metrics/output construction.